In [1]:
# Shared corpus used throughout — semantic groupings make examples clear
DOCS = [
    'the cat sat on the mat',
    'the dog sat on the log',
    'cats and dogs are good animals',
    'the stock market crashed today',
]

In [2]:
import tiktoken

In [3]:
enc = tiktoken.get_encoding('cl100k_base')

In [4]:
ids = enc.encode(DOCS[0])

In [5]:
ids

[1820, 8415, 7731, 389, 279, 5634]

In [6]:
tokens = [enc.decode([t]) for t in ids]

In [7]:
tokens

['the', ' cat', ' sat', ' on', ' the', ' mat']

In [8]:
ids = enc.encode('internationalization')

In [9]:
[enc.decode([t]) for t in ids]

['international', 'ization']

In [10]:
# Bag of Words

In [11]:
tokenized = [doc.lower().split() for doc in DOCS]

In [12]:
vocab = sorted(set([tok for tokens in tokenized for tok in tokens]))

In [13]:
word_to_idx = {w:i for i, w in enumerate(vocab)}

In [14]:
word_to_idx

{'and': 0,
 'animals': 1,
 'are': 2,
 'cat': 3,
 'cats': 4,
 'crashed': 5,
 'dog': 6,
 'dogs': 7,
 'good': 8,
 'log': 9,
 'market': 10,
 'mat': 11,
 'on': 12,
 'sat': 13,
 'stock': 14,
 'the': 15,
 'today': 16}

In [15]:
import numpy as np

In [16]:
bow_matrix = np.zeros((len(DOCS), len(vocab)))

In [17]:
for d_idx, tokens in enumerate(tokenized):
    for tok in tokens:
        bow_matrix[d_idx, word_to_idx[tok]] +=1

In [18]:
bow_matrix

array([[0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 0., 2.,
        0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 1., 1., 0., 2.,
        0.],
       [1., 1., 1., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0.,
        0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 1., 1.,
        1.]])

In [19]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [20]:
cv = CountVectorizer()

In [21]:
X_bow = cv.fit_transform(DOCS)

In [22]:
X_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 21 stored elements and shape (4, 17)>

In [23]:
# TF-IDF

In [ ]:
# BPE

In [24]:
corpus = ['low', 'low', 'lower', 'newest', 'newest', 'newest', 'widest']

In [29]:
tokens = []
for words in corpus:
    tokens += list(words) + ['</w>']

In [50]:
from collections import Counter
pair_count = Counter()
for idx in range(len(tokens)-1):
    if tokens[idx].endswith('</w>'):
        continue
    pair_count[(tokens[idx], tokens[idx+1])] += 1

In [51]:
top_pair, top_freq = pair_count.most_common(1)[0]

In [52]:
new_tokens = []

In [53]:
idx = 0
while idx < len(tokens):
    if idx < len(tokens)-1 and (tokens[idx], tokens[idx+1]) == top_pair:
        new_tokens.append("".join(top_pair))
        idx += 2
    else:
        new_tokens.append(tokens[idx])
        idx+=1

In [69]:
def bpe_tokenizer(corpus, max_iter=10):
    # intialize tokens
    tokens = []
    for words in corpus:
        tokens += list(words) + ['</w>']

    from collections import Counter
    for _ in range(max_iter):
        pair_count = Counter()
        for idx in range(len(tokens)-1):
            if tokens[idx].endswith('</w>'):
                continue
            pair_count[(tokens[idx], tokens[idx+1])] += 1

        top_pair, top_freq = pair_count.most_common(1)[0]

        new_tokens = []
        idx = 0
        while idx < len(tokens):
            if idx < len(tokens)-1 and (tokens[idx], tokens[idx+1]) == top_pair:
                new_tokens.append("".join(top_pair))
                idx += 2
            else:
                new_tokens.append(tokens[idx])
                idx += 1
        tokens = new_tokens
    return tokens

In [70]:
tokens = bpe_tokenizer(corpus)

In [74]:
set(tokens)

{'</w>', 'd', 'e', 'i', 'low</w>', 'lowe', 'newest</w>', 'r', 'st</w>', 'w'}

In [75]:
def bpe_tokenizer_2(corpus, token_limit=10):
    # intialize tokens
    tokens = []
    for words in corpus:
        tokens += list(words) + ['</w>']

    token_count = len(set(tokens))

    from collections import Counter
    while token_count >= token_limit:
        pair_count = Counter()
        for idx in range(len(tokens)-1):
            if tokens[idx].endswith('</w>'):
                continue
            pair_count[(tokens[idx], tokens[idx+1])] += 1

        top_pair, top_freq = pair_count.most_common(1)[0]

        new_tokens = []
        idx = 0
        while idx < len(tokens):
            if idx < len(tokens)-1 and (tokens[idx], tokens[idx+1]) == top_pair:
                new_tokens.append("".join(top_pair))
                idx += 2
            else:
                new_tokens.append(tokens[idx])
                idx += 1
        tokens = new_tokens
        token_count = len(set(tokens))
    return tokens

In [76]:
tokens2 = bpe_tokenizer_2(corpus)

In [77]:
set(tokens2)

{'</w>', 'd', 'e', 'i', 'low</w>', 'lower', 'newest</w>', 'st</w>', 'w'}

In [78]:
tokens2

['low</w>',
 'low</w>',
 'lower',
 '</w>',
 'newest</w>',
 'newest</w>',
 'newest</w>',
 'w',
 'i',
 'd',
 'e',
 'st</w>']